### Nabla without Optim

In [1]:
model_name = "ragdoll"

import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

image = torch.randn(1, 3, 224, 224)
image_np = image.detach().cpu().numpy()
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
output = model(image)
grad = torch.randn_like(output)
grad_np = grad.cpu().numpy()

BENCHMARK_REPEAT=31

In [2]:
#recompute, storeall = [
#    ragdoll.compile(x, "gpu", "input", "codegen", benchmark=True) for x in ["recompute.mlir", "storeall.mlir"]
#]
#recompute = ragdoll.compile("recompute.mlir", "gpu", "input", "codegen", benchmark=True)
#storeall = ragdoll.compile("storeall.mlir", "gpu", "input", "codegen", benchmark=True)
#--iree-hal-benchmark-dispatch-repeat-count=17 \
!iree-compile recompute.mlir \
-o recompute.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

!iree-compile storeall.mlir \
-o storeall.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86
recompute = "recompute.vmfb"
storeall = "storeall.vmfb"

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

#recompute_fb, storeall_fb = [
#    load_executable(x) for x in [recompute, storeall]
#]
recompute_fb = load_executable("recompute.vmfb")
storeall_fb = load_executable("storeall.vmfb")

recompute.mlir:44:11: error: 'tosa.add' op result type '1x197x768' not broadcast compatible with broadcasted operands's shapes '1x197x1'
    %35 = "tosa.add"(%34, %13) : (tensor<1x197x1xf32>, tensor<f32>) -> tensor<1x197x768xf32>
          ^
recompute.mlir:44:11: note: see current operation: %37 = "tosa.add"(%36, %13) : (tensor<1x197x1xf32>, tensor<f32>) -> tensor<1x197x768xf32>
recompute.mlir:749:11: error: 'tosa.add' op result type '1x197x768' not broadcast compatible with broadcasted operands's shapes '1x197x1'
    %40 = "tosa.add"(%39, %7) : (tensor<1x197x1xf32>, tensor<f32>) -> tensor<1x197x768xf32>
          ^
recompute.mlir:749:11: note: see current operation: %43 = "tosa.add"(%42, %7) : (tensor<1x197x1xf32>, tensor<f32>) -> tensor<1x197x768xf32>
storeall.mlir:292:11: error: 'tosa.add' op result type '1x197x768' not broadcast compatible with broadcasted operands's shapes '1x197x1'
    %35 = "tosa.add"(%34, %8) : (tensor<1x197x1xf32>, tensor<f32>) -> tensor<1x197x768xf32>
       

FileNotFoundError: [Errno 2] No such file or directory: 'recompute.vmfb'

In [ ]:
f1 = timeit("recompute_fb.forward(image_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)

"""
f1 = ragdoll_model_benchmark(
    recompute_fb,
    "forward",
    [(1, 3, 224, 224)], 
    device='gpu',
    warmups=2,
    repetitions=BENCHMARK_REPEAT, 
    measure_count=11)
print('ragdoll-opt1-gpu-forward in benchmark: ', f1)
f1 = np.mean(f1)
"""
print(grad_np.shape)
b1 = timeit("recompute_fb.dforward(grad_np)") /  BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-backward in timeit: ', b1)
# TODO(albert): unresolved value error

"""
b1 = ragdoll_model_benchmark(
    recompute,
    "dforward",
    [(1, 1000)],
    device='gpu',
    warmups=2,
    repetitions=BENCHMARK_REPEAT, 
    measure_count=11)
b1 = np.mean(b1)
print('ragdoll-opt1-gpu-backward in benchmark: ', b1)
"""

df = pd.DataFrame()
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-opt1")])
print(df)

### Nabla with Optim

In [ ]:
f1 = timeit("storeall_fb.forward(image_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)

#f2 = ragdoll_model_benchmark(
#    storeall,
#    "forward",
#    [(1, 3, 224, 224)], 
#    device='gpu',
#    warmups=2,
#    repetitions=17, 
#    measure_count=11)
#print('ragdoll-opt1-gpu-forward in benchmark: ', f2)

b1 = timeit("storeall_fb.dforward(grad_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', b1)


df = pd.concat([df, get_dataframe(f1, b1, "Nabla-opt2")])
print(df)

In [ ]:
df.style.hide(axis="index")

In [ ]:
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")

In [ ]:
#baseline is torch_dynamo
baseline_f = f1
baseline_b = b1

forward = df[df["pass"] == "Forward"]
forward["acceleration"] = baseline_f / forward["time"]
forward

In [ ]:
backward = df[df["pass"] == "Backward"]
backward["acceleration"] = baseline_b / backward["time"]
backward

In [ ]:
full = df[df["pass"] == "Full"]
full["acceleration"] = (baseline_b + baseline_f) / full["time"]
full

In [ ]:
df = pd.concat([forward, backward, full])
df.to_csv(f"{model_name}.csv")

sns.barplot(df, x="pass", y="acceleration", hue="item")
plt.xlabel("Pass")
plt.ylabel("Acceleration")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-acceleration.png")